# Test code for api

In [41]:
import os
import pandas as pd
import re
from CM.utils import *
from CM.keys import *

import zipfile
import numpy as np

In [42]:
database = "SocioMap"
syntax = "R"
dirpath = "./tmp"
template = pd.read_csv("55-template.csv")
driver = getDriver(database)

In [43]:
def transform_variables_r(variables):
    variables["transform"] = variables["transform"].str.replace(
        "~", "!", regex=True)
    variables["transform"] = variables["transform"].str.replace(
        "=", "==", regex=True)
    variables["transform"] = variables["transform"].str.replace(
        "!==", "!=", regex=True)
    variables["transform"] = variables["transform"].str.replace(
        "concat", "paste0", regex=True)
    variables["transform"] = variables["transform"].str.replace(
        r',0\)', ',na.rm = True', regex=True)
    variables["transform"] = variables["transform"].str.replace(
        "in", "%in%", regex=True)
    variables["transform"] = variables["transform"].str.replace(
        "na.rm == T", "na.rm = True", regex=True)
    variables["transform"] = variables["transform"].str.replace(
        "== as.numeric", "= as.numeric", regex=True)
    return variables

In [44]:
def load_r_syntax_template(filename, replacements):
    """ Reads R syntax template and replaces placeholders """
    try:
        with open(filename, "r") as file:
            content = file.read()
        for key, value in replacements.items():
            content = content.replace(key, value)
        return content
    except FileNotFoundError:
        print(f"Error: {filename} not found. Ensure the file exists.")
        return None

In [45]:
def zip_output_files(files, dirpath, zip_filename="output.zip"):
    """ Zip all generated files into a single archive """
    zip_path = os.path.join(dirpath, zip_filename)

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file in files:
            if os.path.exists(file):  # Ensure file exists before adding
                # Store without full path
                zipf.write(file, os.path.basename(file))
                print(f"Added to ZIP: {file}")
            else:
                print(
                    f"Warning: {file} does not exist and was not added to the ZIP.")

    print(f"ZIP file created: {zip_path}")
    return zip_path

In [46]:
if "filePath" not in template.columns:
    raise ValueError(
        "Must upload a list of datasets with the filePath column before generating syntax.")

wd = template.iloc[0]["filePath"]

if re.match(r"^[a-zA-Z]:\\\\", wd) or "\\" in wd:
    print("Detected Windows path. Converting to compatible format...")

    # Convert backslashes to forward slashes
    wd = wd.replace("\\", "/")

    # Ensure proper formatting (R escape sequences)
    wd = wd.replace(" ", "\\ ")  # Escape spaces if needed

template = template.iloc[1:]

# verify CMIDs
cols = ["mergingID", "stackID", "datasetID"]
cols = [col for col in cols if col in template.columns]
CMIDs = list(set(template[cols].values.flatten().tolist()))

check = getQuery(
    """
    UNWIND $CMIDs as cmid
    match (a:DATASET {CMID: cmid})
    return a.CMID as CMID, a.CMName as CMName
    """, driver = driver,
    params={"CMIDs": CMIDs},
    type = "df"
)

missing = set(CMIDs) - set(check["CMID"].tolist())
missing = [str(m) + "\n" for m in missing]

if len(check) != len(CMIDs):
    raise ValueError(
        "Error: One or more CMIDs not found in the database\nMissing CMIDs: ", missing)
else:
    print("All CMIDs found in the database.")

Detected Windows path. Converting to compatible format...
All CMIDs found in the database.


In [47]:
db_query = """
    unwind $rows as row
    match (m:DATASET {CMID: row.mergingID})-[rs:MERGING]->(s:DATASET {CMID: row.stackID})-[rm:MERGING]->(v:VARIABLE)<-[ru:USES]-(d:DATASET {CMID: row.datasetID})
    return 
    m.CMID as mergingID, m.CMName as mergingName, s.CMID as stackID, s.CMName as stackName, d.CMID as datasetID, d.CMName as datasetName, rs.aggBy as aggBy, v.CMID as variableCMID, head(apoc.coll.flatten(collect(rm.varName),true)) as varName, rm.transform as transform, rm.Rtransform as Rtransform, rm.Rfunction as Rfunction, rm.summaryStatistic as summaryStatistic, ru.Key as Key
    """
data = getQuery(db_query, driver=driver, params={
                "rows": template.to_dict(orient='records')}, type="df")

pd.set_option('display.max_rows', None)

print(data.head(10)) 


  mergingID                   mergingName stackID  \
0      SD55  Hruschka et al. 2015 (Women)  SD2178   
1      SD55  Hruschka et al. 2015 (Women)  SD2178   
2      SD55  Hruschka et al. 2015 (Women)  SD2178   
3      SD55  Hruschka et al. 2015 (Women)  SD2178   
4      SD55  Hruschka et al. 2015 (Women)  SD2178   
5      SD55  Hruschka et al. 2015 (Women)  SD2178   
6      SD55  Hruschka et al. 2015 (Women)  SD2178   
7      SD55  Hruschka et al. 2015 (Women)  SD2178   
8      SD55  Hruschka et al. 2015 (Women)  SD2178   
9      SD55  Hruschka et al. 2015 (Women)  SD2178   

                                stackName datasetID  \
0  Hruschka et al. 2015 (Women) DHS Stack      SD61   
1  Hruschka et al. 2015 (Women) DHS Stack      SD61   
2  Hruschka et al. 2015 (Women) DHS Stack      SD61   
3  Hruschka et al. 2015 (Women) DHS Stack      SD61   
4  Hruschka et al. 2015 (Women) DHS Stack      SD61   
5  Hruschka et al. 2015 (Women) DHS Stack      SD61   
6  Hruschka et al. 2015 (Women)

In [48]:
if "transform" in data.columns:
    print("transforming variables")

    if "Rtransform" in data.columns:
        data['transform'] = data['Rtransform'].combine_first(
            data['transform'])
        data = transform_variables_r(data)

transforming variables


In [ ]:
data = data.reset_index()

In [ ]:

variables = data[["datasetID","Key"]].copy()
variables = variables.drop_duplicates()
# variables = extract_key(variables, col="Key")
variables[['variable', 'value']] = variables['Key'].str.split(': ', n=1,expand=True)

In [50]:

data = pd.merge(data, variables, on=["datasetID","Key"], how="left")
data["variable"] = data["variable"].str.lower()
data = data.astype(str)
data.replace("None", np.nan, inplace=True)

/tmp/ipykernel_2062036/3098063138.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace("None", np.nan, inplace=True)


In [51]:
data = pd.merge(data, template[["datasetID", "filePath"]], on="datasetID", how="left")

In [52]:
print(dirpath)
data.to_excel(os.path.join(dirpath, "data.xlsx"), index=False)

./tmp


In [53]:
cat_query = """
    unwind $rows as row
    match (d:DATASET {CMID: row.datasetID})-[ru:USES]->(c:CATEGORY) optional match (c)-[:EQUIVALENT]->(e:CATEGORY)
    return d.CMID as datasetID, ru.Key as Key, c.CMID as CMID, c.CMName as CMName, e.CMID as equivalentCMID, e.CMName as equivalentCMName
"""
categories = getQuery(cat_query, driver=driver, params={
                "rows": template.to_dict(orient='records')}, type="df")

print(len(categories))
# number of categories that have equivalent categories
print(len(categories[categories["equivalentCMID"].notnull()]))

10189
519


In [54]:
categories.columns

Index(['datasetID', 'Key', 'CMID', 'CMName', 'equivalentCMID',
       'equivalentCMName'],
      dtype='object')

In [55]:
keys_df = categories[["datasetID", "Key"]].copy()
keys_df = keys_df.drop_duplicates()
# keys_df = extract_key(keys_df, col="Key")
# keys_df = keys_df.melt(
#     id_vars=["datasetID",'Key'],
#     var_name='variable',
#     value_name='value'
# )
# print(keys_df.head(10))
keys_df[['variable', 'value']] = keys_df['Key'].str.split(': ', n=1,expand=True)

In [56]:
categories = pd.merge(categories, keys_df, on=["datasetID","Key"], how="left")
categories = categories.drop_duplicates(subset=["datasetID", "Key", "CMID"])
categories["variable"] = categories["variable"].str.lower()
categories = categories.astype(str)
categories.replace("None", np.nan, inplace=True)

In [57]:
len(categories)
# print(categories.head(100))

9958

In [58]:
categories.to_excel(os.path.join(dirpath, "categories.xlsx"), index=False)

In [106]:
r_syntax_template = "syntax/Rsyntax.txt"
replacements = {
    # Functions applied
    "${f}": "\n".join(data['transform'].dropna()),
    "${wd}": wd,  # Working directory
    "${database}": database  # Database name
}

print(replacements)

{'${f}': '', '${wd}': 'E:/Dropbox\\ (ASU)/RA/CatMapper/Code/SocioMap/data/surveys', '${database}': 'SocioMap'}


In [108]:
r_syntax = load_r_syntax_template(r_syntax_template, replacements)

with open(os.path.join(dirpath, "syntax.R"), "w") as f:
    f.write(r_syntax)
    

In [109]:
files = []
files.extend([
    os.path.join(dirpath, "data.xlsx"),
    os.path.join(dirpath, "categories.xlsx")
])

In [110]:
zip_path = zip_output_files(files, dirpath, "merged_output.zip") 

Added to ZIP: ./tmp/data.xlsx
Added to ZIP: ./tmp/categories.xlsx
ZIP file created: ./tmp/merged_output.zip
